<a href="https://www.kaggle.com/code/fareedhassan/food-prices-forcasting-in-ng?scriptVersionId=366859910" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Forecasting Nigerian Food Prices
Seasonal naive vs SARIMA vs gradient boosting, judged with rolling-origin backtesting.

**Setup:** upload wfp WFP Nigeria food prices file from HDX as a Kaggle dataset and add it as input. (Practice data. Later swap in the real WFP file; see the adapter cell.)

In [ ]:
import glob, warnings, numpy as np, pandas as pd, matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.ensemble import GradientBoostingRegressor
warnings.filterwarnings('ignore')
cands = glob.glob('/kaggle/input/**/*.csv', recursive=True) + glob.glob('*.csv')
print(cands)
path = next(f for f in cands if {'price_ngn_per_kg','price'} & set(pd.read_csv(f, nrows=1).columns))
raw = pd.read_csv(path, low_memory=False)
raw = raw[~raw.iloc[:,0].astype(str).str.startswith('#')]       # drop the HXL tag row in real WFP files
raw['date'] = pd.to_datetime(raw['date'])
if 'price_ngn_per_kg' in raw.columns:                            # synthetic practice file
    df = raw[['date','market','commodity','price_ngn_per_kg']].copy(); df['unit'] = 'KG'
else:                                                            # real WFP / HDX file
    raw['price'] = pd.to_numeric(raw['price'], errors='coerce')
    df = raw[raw.currency=='NGN'].rename(columns={'price':'price_ngn_per_kg'})[['date','market','commodity','unit','price_ngn_per_kg']]
df['month'] = df.date.dt.to_period('M').dt.to_timestamp()
df = df.groupby(['market','commodity','unit','month'], as_index=False).price_ngn_per_kg.mean().rename(columns={'month':'date'})
print(df.shape); print(df.unit.value_counts().head(8))

## 1. Pick a series (market + commodity + unit) with long monthly history

In [ ]:
START = '2015-01-01'          # ignore very old data; change if you want a longer history
d = df[df.date >= START]
last_overall = d.date.max()
cov = (d.groupby(['market','commodity','unit']).date.agg(['count','min','max']).sort_values('count', ascending=False))
recent = cov[cov['max'] >= last_overall - pd.DateOffset(months=6)]     # must still be reporting recently
print('Data runs to', last_overall.date()); print(recent.head(15))
# Auto-pick the best-covered recent rice series. Change MARKET / COMMODITY / UNIT to choose another from the table above.
pick = recent[recent.index.get_level_values('commodity').str.contains('rice', case=False)].index[0]
MARKET, COMMODITY, UNIT = pick
print('Using:', pick)
s = d[(d.market==MARKET)&(d.commodity==COMMODITY)&(d.unit==UNIT)].set_index('date').price_ngn_per_kg
# convert the quoted pack price (e.g. NGN per 2.7 KG) to NGN per kg (or per litre for oils)
import re
def pack_size(unit):
    m = re.match(r'^\s*([\d.]+)?\s*(KG|G|L|ML)\s*$', str(unit).strip(), re.I)
    if not m: return None, str(unit)
    n = float(m.group(1) or 1); u = m.group(2).upper()
    return {'KG':(n,'kg'), 'G':(n/1000,'kg'), 'L':(n,'litre'), 'ML':(n/1000,'litre')}[u]
size, base = pack_size(UNIT)
if size:
    s = s / size; UNIT_LABEL = base
    print(f'Converted from NGN per {UNIT} to NGN per {base}  (divided by {size:g})')
else:
    UNIT_LABEL = UNIT; print('WARNING: unit', UNIT, 'not recognised, prices left as quoted')
series = s.asfreq('MS')
print('months:', len(series), '| missing before cleaning:', series.isna().sum())
# remove obvious recording errors: points far from the local median
med = series.rolling(7, center=True, min_periods=3).median()
bad = (series/med > 1.6) | (series/med < 0.6)
print('outliers removed:', int(bad.sum()))
series = series.mask(bad)
series = series.loc[series.first_valid_index():series.last_valid_index()]
n_filled = int(series.isna().sum())
series = series.interpolate(method='linear').asfreq('MS')
print('months filled by interpolation:', n_filled, 'of', len(series), '| any NaN left:', series.isna().any())
series.plot(title=f'{COMMODITY} price, {MARKET} (NGN per {UNIT_LABEL})'); plt.show()

## 2. Models

In [ ]:
def seasonal_naive(train, h): return np.array([train.iloc[-12 + (i % 12)] for i in range(h)])

def sarima(train, h):
    fit = SARIMAX(np.log(train.values), order=(1,1,1), seasonal_order=(0,1,1,12)).fit(disp=False)
    return np.exp(np.asarray(fit.forecast(h)))

def gboost(train, h):
    s = np.log(train); d = pd.DataFrame({'y': s})
    for l in (1,2,3,6,12): d[f'lag{l}'] = s.shift(l)
    d['m'] = d.index.month; d = d.dropna()
    X, y = d.drop(columns='y'), d.y
    m = GradientBoostingRegressor(random_state=42).fit(X, y)
    hist = list(s.values); idx = train.index[-1]; out = []
    for _ in range(h):
        idx = idx + pd.offsets.MonthBegin(1)
        row = {f'lag{l}': hist[-l] for l in (1,2,3,6,12)}; row['m'] = idx.month
        p = m.predict(pd.DataFrame([row]))[0]; hist.append(p); out.append(p)
    return np.exp(out)
def last_value(train, h): return np.repeat(train.iloc[-1], h)   # naive: next months = this month
models = {'last_value': last_value, 'seasonal_naive': seasonal_naive, 'sarima': sarima, 'gboost': gboost}

## 3. Rolling-origin backtest (12 folds, 6-month horizon)
Train only on the past at each step. MASE below 1 means better than the naive baseline.

In [ ]:
H, FOLDS = 6, 12
scale = np.mean(np.abs(series.values[12:] - series.values[:-12]))   # seasonal naive in-sample error
rows = []
for k in range(FOLDS):
    cut = len(series) - H - (FOLDS-1-k)*3
    train, test = series.iloc[:cut], series.iloc[cut:cut+H]
    for n, f in models.items():
        p = f(train, H); e = test.values - p
        rows.append([n, k, test.index[0].strftime('%Y-%m'), np.mean(np.abs(e)), np.sqrt(np.mean(e**2)), np.mean(np.abs(e))/scale])
res = pd.DataFrame(rows, columns=['model','fold','test_start','MAE','RMSE','MASE'])
summary = res.groupby('model')[['MAE','RMSE','MASE']].mean().round(2).sort_values('MASE'); summary

### Per-fold MAE (NGN)
Shows where each model's errors blow up. Each row is one backtest fold, labelled by the first month it forecasts.

In [ ]:
per_fold = res.pivot_table(index=['fold','test_start'], columns='model', values='MAE').round(0)
per_fold['best'] = per_fold.idxmin(axis=1)
print('Folds won:'); print(per_fold['best'].value_counts())
per_fold

## 4. Plot the last fold

In [ ]:
train, test = series.iloc[:-H], series.iloc[-H:]
ax = series[-36:].plot(label='actual', color='k', figsize=(9,4))
for n,f in models.items():
    pd.Series(f(train,H), index=test.index).plot(ax=ax, marker='o', label=n)   # plot via pandas so the date axis matches
ax.legend(); plt.title('Last 6 months: forecast vs actual'); plt.show()